# VoiceStudio v1: free trial on Google Colab

Try the personal fork (Tanglish ASR through Sarvam, timing lock, presets) on a
**free Colab T4 GPU** before paying for RunPod. The app opens in your browser
through Colab's own proxy: no Mac app, no tunnel, no API key needed for the UI.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the
cells top to bottom. Re-running any cell is safe.

**Optional Colab Secrets** (key icon in the left sidebar, turn *Notebook
access* on). Never paste keys into a cell.

| Secret | Needed for |
| --- | --- |
| `SARVAM_API_KEY` | Tanglish transcription (Sarvam gives free starting credits) |
| `ANTHROPIC_API_KEY` | Claude translation (paid per token). Skip it for a free trial and translate with the offline **Argos** or **NLLB-200** engine |
| `HF_TOKEN` | Only for gated models such as speaker diarization |

**Free-tier limits** (Google changes them without notice): a T4 is not
guaranteed; a session lasts at most about 12 hours and stops if you go idle or
close the tab; when the session ends, everything under `/content` is deleted.
Turn on `USE_DRIVE` below to keep voices and projects in Google Drive.
Models are downloaded again in each new session (about 5–10 minutes).

This notebook reuses cells 1–6 of the upstream notebook
(`notebooks/OmniVoice_Studio_Colab.ipynb`) from your fork's checkout.

In [ ]:
# ── A. Settings ─────────────────────────────────────────────────────────────
FORK_URL = "https://github.com/tn32riderz/VoiceStudio"
BRANCH = "personal/v1"
# Keep voices, projects and exports in Google Drive across sessions
# (asks for Drive access). Models stay on the VM: they would fill a free Drive.
USE_DRIVE = False
# Fallback for a T4 that crashes on the first generation (troubleshooting 5a).
DISABLE_TORCH_COMPILE = False

In [ ]:
# ── B. Clone the fork, load secrets ─────────────────────────────────────────
import os
import subprocess

REPO_DIR = "/content/VoiceStudio"  # the path the upstream cells expect

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"{REPO_DIR} already present; the install cell below updates it.")
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, FORK_URL, REPO_DIR], check=True)
branch = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
if branch != BRANCH:
    raise SystemExit(f"{REPO_DIR} is on {branch!r}, not {BRANCH!r}. Delete the folder "
                     f"(!rm -rf {REPO_DIR}) and re-run this cell.")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["OMNIVOICE_DATA_DIR"] = "/content/drive/MyDrive/voicestudio_trial/omnivoice_data"
    # Exported project bundles too (Change 5; RunPod uses /workspace/projects).
    os.environ["OMNIVOICE_PROJECT_BUNDLE_DIR"] = "/content/drive/MyDrive/voicestudio_trial/projects"
    for path in (os.environ["OMNIVOICE_DATA_DIR"], os.environ["OMNIVOICE_PROJECT_BUNDLE_DIR"]):
        os.makedirs(path, exist_ok=True)
    print("Voices, projects and project bundles are saved in Drive: MyDrive/voicestudio_trial")

if DISABLE_TORCH_COMPILE:
    os.environ["TORCH_COMPILE_DISABLE"] = "1"

# Secrets go into this kernel's environment only; the backend inherits them.
# Values are never printed.
try:
    from google.colab import userdata
except ImportError:
    userdata = None
for name in ("SARVAM_API_KEY", "ANTHROPIC_API_KEY"):
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if value:
        os.environ[name] = value
        print(f"{name}: loaded from Colab Secrets")
    else:
        os.environ.pop(name, None)
        print(f"{name}: not set (optional)")

In [ ]:
# ── C. Install (upstream cells 1-4: GPU check, install, HF token, models) ───
# First run takes about 10-15 minutes.
import json

UPSTREAM_NOTEBOOK = os.path.join(REPO_DIR, "notebooks", "OmniVoice_Studio_Colab.ipynb")
UPSTREAM_CELLS = {
    "gpu": "# ── 1. GPU check",
    "install": "# ── 2. Clone + install",
    "hf_token": "# ── 3. (Optional) Hugging Face token",
    "models": "# ── 4. (Optional, recommended) Pre-download the default models",
    "launch": "# ── 5. Launch the backend",
    "open": "# ── 6. Open the app",
}

def run_upstream(key):
    """Run one upstream notebook cell, found by its heading, in this kernel."""
    with open(UPSTREAM_NOTEBOOK, encoding="utf-8") as f:
        cells = json.load(f)["cells"]
    for cell in cells:
        source = "".join(cell["source"])
        if cell["cell_type"] == "code" and source.startswith(UPSTREAM_CELLS[key]):
            result = get_ipython().run_cell(source)
            if not result.success:
                raise SystemExit(f"Upstream cell {UPSTREAM_CELLS[key]!r} failed; see above.")
            return
    raise SystemExit(f"Cell {UPSTREAM_CELLS[key]!r} is missing from {UPSTREAM_NOTEBOOK}; "
                     "the upstream notebook changed. Update UPSTREAM_CELLS.")

for key in ("gpu", "install", "hf_token", "models"):
    run_upstream(key)

In [ ]:
# ── D. Start the Sarvam ASR proxy (only with SARVAM_API_KEY) ────────────────
import sys
import time
import urllib.request

PROXY_LOG = "/content/sarvam_asr_proxy.log"

def proxy_up():
    try:
        with urllib.request.urlopen("http://127.0.0.1:9876/health", timeout=3) as r:
            return json.load(r).get("configured", False)
    except Exception:
        return False

if not os.environ.get("SARVAM_API_KEY"):
    print("No SARVAM_API_KEY: skipping. Transcription uses the built-in WhisperX.")
elif proxy_up():
    print("Sarvam ASR proxy already running on 127.0.0.1:9876.")
else:
    subprocess.Popen(
        [sys.executable, "-m", "tools.sarvam_asr_proxy", "--host", "127.0.0.1", "--port", "9876"],
        cwd=REPO_DIR, stdout=open(PROXY_LOG, "ab"), stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        if proxy_up():
            print("Sarvam ASR proxy running on 127.0.0.1:9876.")
            break
        time.sleep(1)
    else:
        print(open(PROXY_LOG, errors="replace").read()[-2000:])
        raise SystemExit("The Sarvam proxy did not start; see its log above.")

In [ ]:
# ── E. Launch the backend and open the app (upstream cells 5-6) ─────────────
# Keep this tab open while you work: closing it ends the free session.
run_upstream("launch")
run_upstream("open")

## In the app (browser tab that just opened)

1. Finish the setup wizard.
2. **Tanglish transcription** (only with `SARVAM_API_KEY`):
   **Settings → Models → ASR → OpenAI-compatible (remote server)**:
   Server URL `http://127.0.0.1:9876/v1`, Model `saaras-v4-codemix`, API key
   empty, **Test connection**, then **Use**.
3. **Translation**: with `ANTHROPIC_API_KEY`, pick **Anthropic (Claude)** in
   **Settings → LLM Providers**. For a free trial, choose **Argos** or
   **NLLB-200** as the translation engine instead.
4. **Dub page**: upload a short (1–2 minute) Tanglish clip → Transcribe →
   Translate → Generate → listen → Export.

**What to check before buying RunPod:** the Tamil transcript keeps English
words in English, the dubbed voice sounds like you, every line fits its time
slot, and nothing crashes on the T4. A T4 is slower than RunPod's RTX 4090,
so time per video here is not what RunPod will take.

**If the first generation crashes the backend:** set
`DISABLE_TORCH_COMPILE = True` in cell A, then
`Runtime → Restart session` and run all cells again.